# Lab 1 at home · Two tables of your own in DuckDB

**Due:** Sunday, October 25 at 11:59 PM · **Submit:** your repo URL on CourseWorks

In class you loaded one table (penguins), found a problem and fixed it. Now do the same with **two related tables you choose**. In Session 2 we'll join them, so pick two that can be connected.

You will:
1. Pick two related public datasets and upload them
2. Load both into DuckDB and check what DuckDB decided about each column
3. Find the column the two tables share (the **key**) and check how well they match
4. Write five SQL queries that help explain your data
5. Write observations and a dataset card for each table

## 1. Pick two related tables

**Related** means they share a column you could match rows on: an ID, a name, a ZIP code, a country code, a date.

**Example pairs**
- **World Bank:** any indicator download (e.g. GDP per country per year) comes with a second file of country details (region, income group). The shared column is the country code.
- **Our World in Data:** two charts for the same countries, e.g. life expectancy and health spending. Shared columns: country and year.
- **NYC Open Data:** 311 complaints and population by ZIP code, or the street tree census and neighborhood boundaries. Shared column: ZIP code or neighborhood.
- **Your own idea:** any two public tables that share a column.

**Rules**
- **Public, open data only.** Your repo is public, and later this data goes to a free AI tier. Nothing from work, nothing personal, nothing behind a login.
- **CSV format.** If a file is Excel, open it and save it as CSV first.
- **Each file under 100 MB.** Between 1,000 and 1,000,000 rows is a good size for the main table; the second can be small.
- **Reuse allowed.** Check each license or terms page, and note it in your dataset card.

**Places to look:** [NYC Open Data](https://opendata.cityofnewyork.us) · [data.gov](https://data.gov) · [World Bank Open Data](https://data.worldbank.org) · [Our World in Data](https://ourworldindata.org) · [Kaggle Datasets](https://www.kaggle.com/datasets) (check each license)

## 2. Upload them

1. Download both CSVs to your computer.
2. In this Codespace, drag both files into the **`data`** folder in the file list on the left.
3. Rename each to something short with no spaces, e.g. `gdp.csv` and `countries.csv` (right-click → **Rename**).
4. Fill in the cell below and run it. Table names should be short, lowercase, no spaces: you'll type them in your SQL.

In [2]:
FILE_A  = "life-expectancy.csv"     # <- your main table's file name
TABLE_A = "life_expectancy"                   # <- a short name for it, e.g. "gdp"

FILE_B  = "gdp-per-capita-worldbank.csv"   # <- the related table's file name
TABLE_B = "gdp_per_capita"                 # <- a short name for it, e.g. "countries"

In [3]:
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_A = ROOT / "data" / FILE_A
DATA_B = ROOT / "data" / FILE_B

missing = [p.name for p in (DATA_A, DATA_B) if not p.exists()]
if missing:
    print("Can't find:", ", ".join(missing))
    print("Files in data/ right now:")
    for f in sorted((ROOT / "data").iterdir()):
        print("  ", f.name)
else:
    con = duckdb.connect(str(ROOT / "lab1_home.duckdb"))
    for p in (DATA_A, DATA_B):
        print(f"Found data/{p.name} ({p.stat().st_size / 1_000_000:.1f} MB)")
    print("Connected.")

Found data/life-expectancy.csv (0.6 MB)
Found data/gdp-per-capita-worldbank.csv (0.3 MB)
Connected.


## 3. First look

Preview the first 10 rows of each raw file before loading.

In [4]:
con.sql(f"SELECT * FROM read_csv('{DATA_A}') LIMIT 10")

┌─────────────┬─────────┬───────┬─────────────────┐
│   Entity    │  Code   │ Year  │ Life expectancy │
│   varchar   │ varchar │ int64 │     double      │
├─────────────┼─────────┼───────┼─────────────────┤
│ Afghanistan │ AFG     │  1950 │         28.1563 │
│ Afghanistan │ AFG     │  1951 │         28.5836 │
│ Afghanistan │ AFG     │  1952 │         29.0138 │
│ Afghanistan │ AFG     │  1953 │         29.4521 │
│ Afghanistan │ AFG     │  1954 │         29.6975 │
│ Afghanistan │ AFG     │  1955 │          30.366 │
│ Afghanistan │ AFG     │  1956 │         30.8303 │
│ Afghanistan │ AFG     │  1957 │         31.3451 │
│ Afghanistan │ AFG     │  1958 │           31.84 │
│ Afghanistan │ AFG     │  1959 │         32.3365 │
└─────────────┴─────────┴───────┴─────────────────┘
  10 rows                               4 columns

In [5]:
con.sql(f"SELECT * FROM read_csv('{DATA_B}') LIMIT 10")

┌─────────────┬─────────┬───────┬────────────────┬────────────────────────────────┐
│   Entity    │  Code   │ Year  │ GDP per capita │ World region according to OWID │
│   varchar   │ varchar │ int64 │     double     │            varchar             │
├─────────────┼─────────┼───────┼────────────────┼────────────────────────────────┤
│ Afghanistan │ AFG     │  2000 │      1617.8264 │ Asia                           │
│ Afghanistan │ AFG     │  2001 │      1454.1108 │ Asia                           │
│ Afghanistan │ AFG     │  2002 │      1774.3087 │ Asia                           │
│ Afghanistan │ AFG     │  2003 │      1815.9282 │ Asia                           │
│ Afghanistan │ AFG     │  2004 │      1776.9182 │ Asia                           │
│ Afghanistan │ AFG     │  2005 │      1908.1147 │ Asia                           │
│ Afghanistan │ AFG     │  2006 │      1929.7239 │ Asia                           │
│ Afghanistan │ AFG     │  2007 │       2155.353 │ Asia                     

## 4. Load both and check the types

Load each file into a table, then `DESCRIBE` it. Compare each column's type with what it *should* be. A number or date column stored as `VARCHAR` (text) is the same kind of problem you fixed in class.

In [6]:
for table, path in ((TABLE_A, DATA_A), (TABLE_B, DATA_B)):
    con.sql(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM read_csv('{path}')")
    rows = con.sql(f"SELECT count(*) FROM {table}").fetchone()[0]
    print(f"{table}: {rows:,} rows loaded")

life_expectancy: 21,565 rows loaded
gdp_per_capita: 7,445 rows loaded


In [7]:
con.sql(f"DESCRIBE {TABLE_A}")

┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ Entity          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Code            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Year            │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Life expectancy │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [8]:
con.sql(f"DESCRIBE {TABLE_B}")

┌────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name           │ column_type │  null   │   key   │ default │  extra  │
│            varchar             │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ Entity                         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Code                           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Year                           │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ GDP per capita                 │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ World region according to OWID │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

`SUMMARIZE` profiles every column at once: min, max, how many distinct values, and `null_percentage` (how much is missing).

In [9]:
con.sql(f"SUMMARIZE {TABLE_A}")

┌─────────────────┬─────────────┬─────────────┬──────────┬───────────────┬────────────────────┬───────────────────┬────────────────────┬───────────────────┬─────────────────┬───────┬─────────────────┐
│   column_name   │ column_type │     min     │   max    │ approx_unique │        avg         │        std        │        q25         │        q50        │       q75       │ count │ null_percentage │
│     varchar     │   varchar   │   varchar   │ varchar  │     int64     │      varchar       │      varchar      │      varchar       │      varchar      │     varchar     │ int64 │  decimal(9,2)   │
├─────────────────┼─────────────┼─────────────┼──────────┼───────────────┼────────────────────┼───────────────────┼────────────────────┼───────────────────┼─────────────────┼───────┼─────────────────┤
│ Entity          │ VARCHAR     │ Afghanistan │ Zimbabwe │           295 │ NULL               │ NULL              │ NULL               │ NULL              │ NULL            │ 21565 │            0.

In [10]:
con.sql(f"SUMMARIZE {TABLE_B}")

┌────────────────────────────────┬─────────────┬─────────────┬───────────────┬───────────────┬────────────────────┬───────────────────┬──────────────────┬────────────────────┬────────────────────┬───────┬─────────────────┐
│          column_name           │ column_type │     min     │      max      │ approx_unique │        avg         │        std        │       q25        │        q50         │        q75         │ count │ null_percentage │
│            varchar             │   varchar   │   varchar   │    varchar    │     int64     │      varchar       │      varchar      │     varchar      │      varchar       │      varchar       │ int64 │  decimal(9,2)   │
├────────────────────────────────┼─────────────┼─────────────┼───────────────┼───────────────┼────────────────────┼───────────────────┼──────────────────┼────────────────────┼────────────────────┼───────┼─────────────────┤
│ Entity                         │ VARCHAR     │ Afghanistan │ Zimbabwe      │           212 │ NULL         

## 5. Fix what loaded wrong (if anything)

If every type looks right, write "No fixes needed" in the text cell after the next one and skip ahead.

Otherwise, open the CSV (double-click it in the `data` folder) and find **how missing values are written** in the columns that came in as text. Edit the `nullstr` list below to match, and run it for the table that needs fixing.

In [11]:
# Change TABLE_A / DATA_A to TABLE_B / DATA_B if it's the second table that needs fixing
con.sql(f"""
    CREATE OR REPLACE TABLE {TABLE_A} AS
    SELECT * FROM read_csv('{DATA_A}', nullstr = ['NA', 'N/A', ''])
""")
con.sql(f"DESCRIBE {TABLE_A}")

┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ Entity          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Code            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Year            │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Life expectancy │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

**What did you fix, and how did you know?** (Double-click to edit.)
No fixes needed. I checked both tables using DESCRIBE. All columns have the expected data types. Country names and codes are stored as VARCHAR, years as BIGINT, and life expectancy and GDP per capita as DOUBLE. Therefore, no type corrections were necessary.



## 6. Find the key: how do the two tables connect?

The **key** is the column both tables share. Type its name in each table below. It may be spelled differently in each, e.g. `Country Code` in one and `iso3` in the other.

The next cell counts the distinct key values in each table and how many of them appear in both. (It treats both as text, so a ZIP code stored as a number still matches one stored as text.)

In [12]:
KEY_A = "Code"   # <- the shared column's name in your main table
KEY_B = "Code"   # <- its name in the second table

In [13]:
con.sql(f"""
    WITH a AS (SELECT DISTINCT CAST("{KEY_A}" AS VARCHAR) AS k FROM {TABLE_A}),
         b AS (SELECT DISTINCT CAST("{KEY_B}" AS VARCHAR) AS k FROM {TABLE_B})
    SELECT
        (SELECT count(*) FROM a)                           AS keys_in_a,
        (SELECT count(*) FROM b)                           AS keys_in_b,
        (SELECT count(*) FROM a WHERE k IN (SELECT k FROM b)) AS keys_in_both
""")

┌───────────┬───────────┬──────────────┐
│ keys_in_a │ keys_in_b │ keys_in_both │
│   int64   │   int64   │    int64     │
├───────────┼───────────┼──────────────┤
│       248 │       212 │          204 │
└───────────┴───────────┴──────────────┘

In [14]:
con.sql("""
    SELECT DISTINCT
        a.Entity,
        a.Code
    FROM life_expectancy a
    WHERE a.Code IS NOT NULL
      AND a.Code NOT IN (
          SELECT Code
          FROM gdp_per_capita
          WHERE Code IS NOT NULL
      )
    ORDER BY a.Entity
    LIMIT 10
""")

┌─────────────────────────────────┬──────────┐
│             Entity              │   Code   │
│             varchar             │ varchar  │
├─────────────────────────────────┼──────────┤
│ Africa                          │ OWID_AFR │
│ American Samoa                  │ ASM      │
│ Anguilla                        │ AIA      │
│ Asia                            │ OWID_ASI │
│ Bonaire Sint Eustatius and Saba │ BES      │
│ British Virgin Islands          │ VGB      │
│ Cook Islands                    │ COK      │
│ Cuba                            │ CUB      │
│ Eritrea                         │ ERI      │
│ Europe                          │ OWID_EUR │
└─────────────────────────────────┴──────────┘
  10 rows                          2 columns

**Describe the relationship** (double-click to edit):

- **Key column:** In the main table, it is called Code, and in the second table, it is also called Code.
- **How they relate:** Both tables contain data for different countries and years. Each country has multiple records, one for each year, so matching only by Code can produce a many-to-many relationship. I would use Code and Year together to match the same country in the same year.
- **How well they match:** The life expectancy table has 248 distinct Code values(including NULL), while the GDP per capita table has 212. Among them, 204 appear in both tables. There are 44 unmatched values in the first table(including NULL) and 8 in the second. For example, Africa (OWID_AFR) appears only in the life expectancy table, probably because the two datasets use different regional classifications.



## 7. Five queries that explain your data

Write one query of each kind. Under each, double-click the text cell and write one line: **what does this result tell us?** Use either table.

Tip: if a column name has spaces or capitals, wrap it in double quotes: `"Country Name"`. Start from a query in the in-class notebook and change one thing.

### Query 1 · Size and shape
*How many rows, and over what time period or area?* Useful SQL: `count(*), min(), max()`

In [15]:
con.sql("""
    -- write your query here; use your own table names
    SELECT
        COUNT(*) AS total_rows,
        MIN(Year) AS first_year,
        MAX(Year) AS last_year
    FROM life_expectancy
""")

┌────────────┬────────────┬───────────┐
│ total_rows │ first_year │ last_year │
│   int64    │   int64    │   int64   │
├────────────┼────────────┼───────────┤
│      21565 │       1543 │      2023 │
└────────────┴────────────┴───────────┘

**What this tells us:** The life expectancy dataset contains 21,565 rows, covering the period from 1543 to 2023. However, not every country has data for the entire period.

### Query 2 · Filter
*Which rows meet a condition you care about?* Useful SQL: `WHERE`

In [16]:
con.sql("""
    -- write your query here; use your own table names
    SELECT
        Entity,
        Year,
        "Life expectancy"
    FROM life_expectancy
    WHERE Code = 'CHN'
      AND Year >= 2000
    ORDER BY Year
""")

┌─────────┬───────┬─────────────────┐
│ Entity  │ Year  │ Life expectancy │
│ varchar │ int64 │     double      │
├─────────┼───────┼─────────────────┤
│ China   │  2000 │         72.2885 │
│ China   │  2001 │         72.6755 │
│ China   │  2002 │         73.0339 │
│ China   │  2003 │         73.3853 │
│ China   │  2004 │         73.7274 │
│ China   │  2005 │          74.085 │
│ China   │  2006 │         74.4037 │
│ China   │  2007 │         74.7886 │
│ China   │  2008 │         74.8333 │
│ China   │  2009 │         75.3174 │
│   ·     │    ·  │            ·    │
│   ·     │    ·  │            ·    │
│   ·     │    ·  │            ·    │
│ China   │  2014 │         76.7143 │
│ China   │  2015 │         76.9755 │
│ China   │  2016 │          77.214 │
│ China   │  2017 │         77.2353 │
│ China   │  2018 │          77.709 │
│ China   │  2019 │         77.9396 │
│ China   │  2020 │         78.0191 │
│ China   │  2021 │         78.1166 │
│ China   │  2022 │         78.2021 │
│ China   │ 

**What this tells us:** Life expectancy in China increased from 72.29 years in 2000 to 77.95 years in 2023, an increase of about 5.66 years. However, there was a slight decrease from 2022 to 2023.

### Query 3 · Group
*How does a count or average differ across categories?* Useful SQL: `GROUP BY`

In [17]:
con.sql("""
    -- write your query here; use your own table names
    SELECT
        "World region according to OWID" AS region,
        COUNT(*) AS country_count,
        ROUND(AVG("GDP per capita"), 2) AS avg_gdp
    FROM gdp_per_capita
    WHERE Year = 2020
      AND "World region according to OWID" IS NOT NULL
      AND "GDP per capita" IS NOT NULL
    GROUP BY region
    ORDER BY avg_gdp DESC
""")

┌───────────────┬───────────────┬──────────┐
│    region     │ country_count │ avg_gdp  │
│    varchar    │     int64     │  double  │
├───────────────┼───────────────┼──────────┤
│ Europe        │            44 │ 45916.39 │
│ North America │            31 │ 28399.45 │
│ Asia          │            47 │ 26376.72 │
│ South America │            11 │ 18251.06 │
│ Oceania       │            14 │ 13340.57 │
│ Africa        │            52 │  6477.16 │
└───────────────┴───────────────┴──────────┘

**What this tells us:** The results show clear differences in average GDP per capita across world regions in 2020. Europe had the highest average GDP per capita ($45,916.39), while Africa had the lowest ($6,477.16). This suggests a large economic gap between regions.

### Query 4 · Rank
*What are the top or bottom 10 of something?* Useful SQL: `ORDER BY ... LIMIT 10`

In [18]:
con.sql("""
    -- write your query here; use your own table names
     SELECT
        Entity,
        "GDP per capita"
    FROM gdp_per_capita
    WHERE Year = 2020
      AND "World region according to OWID" IS NOT NULL
      AND "GDP per capita" IS NOT NULL
    ORDER BY "GDP per capita" DESC
    LIMIT 10
""")

┌────────────────┬────────────────┐
│     Entity     │ GDP per capita │
│    varchar     │     double     │
├────────────────┼────────────────┤
│ Luxembourg     │      129865.63 │
│ Singapore      │      116684.21 │
│ Qatar          │     103061.914 │
│ Ireland        │     101793.445 │
│ Norway         │       89012.27 │
│ Bermuda        │       88484.81 │
│ Brunei         │        80280.9 │
│ Switzerland    │      79270.875 │
│ Cayman Islands │       72183.13 │
│ Greenland      │      68533.734 │
└────────────────┴────────────────┘
  10 rows               2 columns

**What this tells us:** The results show that Luxembourg had the highest GDP per capita in 2020 (129,865.63), followed by Singapore and Qatar. Several small countries and territories also ranked among the top 10.

### Query 5 · Quality check
*What's missing, duplicated or impossible?* Useful SQL: `count(*) - count(column), count(DISTINCT ...)`

In [19]:
con.sql("""
    -- write your query here; use your own table names
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(Code) AS missing_codes,
        COUNT(*) - COUNT("Life expectancy") AS missing_life_expectancy,
        SUM(
            CASE
                WHEN "Life expectancy" <= 0
                  OR "Life expectancy" > 120
                THEN 1
                ELSE 0
            END
        ) AS unusual_values
    FROM life_expectancy
""")

┌────────────┬───────────────┬─────────────────────────┬────────────────┐
│ total_rows │ missing_codes │ missing_life_expectancy │ unusual_values │
│   int64    │     int64     │          int64          │     int128     │
├────────────┼───────────────┼─────────────────────────┼────────────────┤
│      21565 │          1348 │                       0 │              0 │
└────────────┴───────────────┴─────────────────────────┴────────────────┘

**What this tells us:** The life expectancy dataset contains 21,565 records, including 1,348 records with missing country codes. However, there are no missing life expectancy values or unusual values based on my checks. The missing country codes may affect how the two datasets are matched.

## 8. Three observations

Specific things you noticed that you'd want to check before trusting this data. Point to numbers.

1. The life expectancy dataset has 1,348 records with missing country codes out of 21,565 rows. This may affect how the two datasets are matched.
2. The two datasets share 204 country code values, but 44 code values appear only in the life expectancy dataset and 8 only in the GDP dataset. I would like to check why some countries or regions are missing from one dataset.
3. The life expectancy dataset covers a long period from 1543 to 2023. However, not every country has data for the entire period. I would like to check the availability and reliability of the historical data.

**One question you'd want a non-technical person to be able to ask of this data:** How does life expectancy differ between countries with higher and lower GDP per capita?



## 9. Dataset cards

| | Main table | Second table |
|---|---|---|
| **Dataset name** |Life Expectancy |GDP per Capita |
| **Source (link)** | Our World in Data|Our World in Data |
| **Who collected it, and why** |Data from Riley, Zijdeman, HMD, and the UN, combined by OWID to study long-term changes in life expectancy. | Data from Eurostat, OECD, IMF, and the World Bank, processed by OWID to compare economic development across countries.|
| **License or terms of use** |OWID: CC BY. Original data providers' terms also apply. | OWID: CC BY. World Bank WDI: CC BY 4.0, with applicable source-specific terms.|
| **Rows × columns** |21,565 × 4 | 7,445 × 5|
| **Time period covered** |1543–2023 |1990–2025 |
| **How often it's updated** |Approximately annually. Last updated October 22, 2025; next expected update November 25, 2026. |Periodically. Last updated July 2026; next expected update January 2027. |

## 10. Submit

1. Run the cell below to close the connection.
2. Fill in the lifecycle map of your data in `labs/lab1.md`.
3. **Save this notebook (Cmd+S)**, then commit and push: **Source Control** → message `Lab 1` → **Commit** → **Sync Changes**.
4. Check on GitHub that both CSVs, both notebooks and `labs/lab1.md` are there, with results visible.
5. Paste your repo URL into the Lab 1 assignment on CourseWorks.

In [20]:
con.close()
print("Done. Save, commit, push, and submit your repo URL.")

Done. Save, commit, push, and submit your repo URL.
